# Step 6 — Does the palm-oil price predict forest loss?

The economic half of the project. We pair **annual forest loss** in the AOI with the **global palm-oil price** and test whether price moves predict clearing — possibly with a lag (clear land when palm is profitable, harvest later).

**Why Hansen, not our U-Net here:** Step 5 showed a 3-snapshot model difference is too noisy and conflates plantation with forest. **Hansen Global Forest Change** gives a consistent *annual* record of tree-cover **loss events** for 2001–2023 — ~23 data points, which is what makes a regression meaningful.

No GPU or Drive needed — runs top to bottom in one kernel.

In [ ]:
!pip install -q earthengine-api statsmodels
import ee, numpy as np, pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project="landcover-riau")

## 1. Annual forest loss from Hansen (2001–2023)

The `lossyear` band encodes the year each pixel lost tree cover (1 = 2001 … 23 = 2023). We restrict to pixels that were real forest in 2000 (`treecover2000 ≥ 30%`), then sum the lost **area** grouped by year.

In [ ]:
AOI_BBOX = [102.9, -0.6, 103.4, -0.1]
region = ee.Geometry.Rectangle(AOI_BBOX)
TC_THRESHOLD = 30        # % canopy in 2000 to count as 'forest'

gfc = ee.Image("UMD/hansen/global_forest_change_2023_v1_11").clip(region)
forest2000 = gfc.select("treecover2000").gte(TC_THRESHOLD)
loss = gfc.select("loss").And(forest2000)
lossyear = gfc.select("lossyear").updateMask(loss)
area = ee.Image.pixelArea().updateMask(loss)        # m^2 of lost forest

groups = area.addBands(lossyear).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="yr"),
    geometry=region, scale=30, maxPixels=1e10,
).getInfo()["groups"]

loss_df = (pd.DataFrame([{"year": 2000 + g["yr"], "loss_km2": g["sum"] / 1e6} for g in groups])
           .query("year >= 2001").sort_values("year").reset_index(drop=True))
print(loss_df.to_string(index=False))
print(f"\ntotal forest lost 2001-2023: {loss_df.loss_km2.sum():.1f} km^2")

## 2. Global palm-oil price

Annual average of the global palm-oil price (USD / metric ton), from FRED series `PPALPLUSDM` (IMF / World Bank primary-commodity prices). FRED serves a stable CSV, so this just works — no manual download.

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=PPALPLUSDM"
raw = pd.read_csv(url)
raw.columns = ["date", "palm_usd"]                      # force names (FRED header varies)
raw["date"] = pd.to_datetime(raw["date"])
raw["palm_usd"] = pd.to_numeric(raw["palm_usd"], errors="coerce")
raw["year"] = raw["date"].dt.year
price = raw.groupby("year")["palm_usd"].mean().rename("palm_usd")
print(price.loc[2001:2023].round(0).to_string())

## 3. Lagged regression

Merge the two series and regress forest loss on the palm price at several lags: does this year's clearing track the price 0, 1, 2, or 3 years earlier?

In [ ]:
df = loss_df.set_index("year").join(price).dropna()

results = {}
print("OLS:  forest_loss ~ palm_price(t - lag)")
for lag in range(0, 4):
    d = df.assign(price_lag=df["palm_usd"].shift(lag)).dropna()
    res = sm.OLS(d["loss_km2"], sm.add_constant(d["price_lag"])).fit()
    results[lag] = (res, d)
    print(f"  lag {lag}y:  n={int(res.nobs)}  R2={res.rsquared:.3f}  "
          f"slope={res.params['price_lag']:+.4f}  p={res.pvalues['price_lag']:.3f}")

best_lag = max(results, key=lambda L: results[L][0].rsquared)
print(f"\nbest fit at lag = {best_lag} year(s)")

## 4. Plots

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(df.index, df["loss_km2"], color="#d7191c", alpha=0.55)
ax1.set_ylabel("forest loss (km²)", color="#d7191c"); ax1.set_xlabel("year")
ax2 = ax1.twinx()
ax2.plot(df.index, df["palm_usd"], "o-", color="#1b7837")
ax2.set_ylabel("palm-oil price ($/mt)", color="#1b7837")
ax1.set_title("Annual forest loss vs palm-oil price — Indragiri AOI")
plt.tight_layout(); plt.show()

In [ ]:
res, d = results[best_lag]
xs = np.linspace(d["price_lag"].min(), d["price_lag"].max(), 50)
plt.figure(figsize=(5.5, 5))
plt.scatter(d["price_lag"], d["loss_km2"], color="#2c7fb8")
plt.plot(xs, res.params["const"] + res.params["price_lag"] * xs, color="red")
plt.xlabel(f"palm-oil price, t − {best_lag} yr ($/mt)"); plt.ylabel("forest loss (km²)")
plt.title(f"lag {best_lag}y:  R²={res.rsquared:.2f},  p={res.pvalues['price_lag']:.3f}")
plt.tight_layout(); plt.show()

## 5. Read-out

Interpret honestly:

- **Sign + strength:** a positive slope with the best R² at a 1–2 yr lag would support "high prices → clearing, with a delay." A weak/insignificant fit is also a real result — say so.
- **Caveats (state these):** only ~23 yearly points; OLS p-values are optimistic on autocorrelated time series; the price is *global* while loss is one small AOI; and big confounders exist (the 2011 forest moratorium, the 2015 fire/haze year, COVID, replanting cycles). So this is **association, not causation**.
- **The pairing is the point:** the U-Net gives the high-detail land-cover maps (the CV showcase); Hansen + price gives the long, testable economic signal. Two methods, two scales, one story.

**Project status:** this closes the analytical loop. Remaining polish = the write-up — drop the key figures into the README, add a short methods/limitations note, and (optional extension) the oil-palm layer to split natural forest from plantation.